# HRMS Database Query Testing

This notebook opens the HRMS SQLite database in read-only mode. Run the connection cell first, then list tables, inspect a schema, and write a SELECT or WITH query in the query cell. Results are capped at 100 rows.

In [ ]:
import re
import sqlite3
from pathlib import Path

search_roots = []
for base in (Path.cwd(), *Path.cwd().parents):
    search_roots.extend((base, base / "HRMS-main"))
project_root = next(
    (root for root in search_roots if (root / "backend" / "app" / "main.py").is_file()),
    None,
)
if project_root is None:
    backend_root = next(
        (root for root in search_roots if (root / "app" / "main.py").is_file()),
        None,
    )
    if backend_root is not None:
        project_root = backend_root.parent
if project_root is None:
    raise FileNotFoundError("Could not locate HRMS-main/backend/app/main.py from the notebook working directory.")

database_path = project_root / "backend" / "storage" / "hrms.db"
if not database_path.is_file():
    raise FileNotFoundError(f"HRMS database not found: {database_path}")

database_uri = f"{database_path.as_uri()}?mode=ro"
connection = sqlite3.connect(database_uri, uri=True)
connection.row_factory = sqlite3.Row

def allow_read_only_sql(action, argument1, argument2, database_name, source):
    if action in {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_RECURSIVE, sqlite3.SQLITE_READ}:
        return sqlite3.SQLITE_OK
    if action == sqlite3.SQLITE_FUNCTION:
        function_name = (argument2 or "").casefold()
        if function_name not in {"load_extension", "readfile", "writefile"}:
            return sqlite3.SQLITE_OK
    return sqlite3.SQLITE_DENY

connection.set_authorizer(allow_read_only_sql)
print(f"Connected read-only to {database_path}")

## List database tables

Lists application tables and excludes SQLite's internal tables.

In [ ]:
tables = connection.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
).fetchall()
table_names = [row["name"] for row in tables]
for table_name in table_names:
    print(table_name)
print(f"\nTotal application tables: {len(table_names)}")

## Inspect a table schema

Set `table_name` to a name printed above to show its CREATE TABLE statement. Schema output may include sensitive column names.

In [ ]:
table_name = "employees"
if table_name not in table_names:
    raise ValueError(f"Unknown table: {table_name}. Run the table-listing cell first.")

definition = connection.execute(
    "SELECT sql FROM sqlite_master WHERE type = 'table' AND name = ?",
    (table_name,),
).fetchone()
if definition is None:
    raise ValueError(f"Schema not found for table: {table_name}")
print(definition["sql"])

## Write and run a query

Edit `query` and run this cell. The connection is read-only; only one SELECT or WITH query is accepted. The cell displays at most 100 result rows.

In [ ]:
# query = """
# SELECT status, COUNT(*) AS employee_count
# FROM employees
# GROUP BY status
# ORDER BY status
# """.strip()

# query = """
# SELECT employees.name 
# FROM employees 
# JOIN employee_skills ON employees.id = employee_skills.employee_id 
# JOIN skills ON employee_skills.skill_id = skills.id 
# WHERE skills.name IN ('Python', 'FastAPI') 
# GROUP BY employees.id
# """.strip()

# query = """
# SELECT employees.id, employees.name, employees.email 
# FROM employees 
# JOIN employee_skills ON employees.id = employee_skills.employee_id 
# JOIN skills ON employee_skills.skill_id = skills.id 
# WHERE skills.name = 'Python';
# """.strip()

query = """
select manager_id from employees where id = 1;
""".strip()

if not re.match(r"^(SELECT|WITH)\b", query, flags=re.IGNORECASE):
    raise ValueError("Only SELECT or WITH queries are allowed.")
if ";" in query.rstrip().removesuffix(";"):
    raise ValueError("Only one SQL statement is allowed.")

try:
    cursor = connection.execute(query)
    if cursor.description is None:
        raise ValueError("The query did not return rows. Use a SELECT query.")
    result_rows = cursor.fetchmany(101)
    truncated = len(result_rows) > 100
    result_rows = result_rows[:100]
    print("Columns:", [description[0] for description in cursor.description])
    for result_row in result_rows:
        print(dict(result_row))
    print()
    print(f"Rows shown: {len(result_rows)}")
    if truncated:
        print("Results were truncated to the first 100 rows.")
except sqlite3.Error as error:
    print(f"SQLite error: {error}")

In [ ]:
# query = """
# SELECT status, COUNT(*) AS employee_count
# FROM employees
# GROUP BY status
# ORDER BY status
# """.strip()

# query = """
# SELECT employees.name 
# FROM employees 
# JOIN employee_skills ON employees.id = employee_skills.employee_id 
# JOIN skills ON employee_skills.skill_id = skills.id 
# WHERE skills.name IN ('Python', 'FastAPI') 
# GROUP BY employees.id
# """.strip()

query = """
with e as (
select distinct id, name, role from employees
),

es as (
select distinct employee_id, skill_id from employee_skills
),
s as 
(select distinct id, name from skills where name = "Python")

select e.id, e.name, e.role, s.name 
from e inner join es on e.id = es.employee_id
inner join s on 
es.skill_id = s.id;





""".strip()

if not re.match(r"^(SELECT|WITH)\b", query, flags=re.IGNORECASE):
    raise ValueError("Only SELECT or WITH queries are allowed.")
if ";" in query.rstrip().removesuffix(";"):
    raise ValueError("Only one SQL statement is allowed.")

try:
    cursor = connection.execute(query)
    if cursor.description is None:
        raise ValueError("The query did not return rows. Use a SELECT query.")
    result_rows = cursor.fetchmany(101)
    truncated = len(result_rows) > 100
    result_rows = result_rows[:100]
    print("Columns:", [description[0] for description in cursor.description])
    for result_row in result_rows:
        print(dict(result_row))
    print()
    print(f"Rows shown: {len(result_rows)}")
    if truncated:
        print("Results were truncated to the first 100 rows.")
except sqlite3.Error as error:
    print(f"SQLite error: {error}")

## Close the connection

Run when finished. Re-run the connection cell before executing more queries.

In [ ]:
connection.close()
print("Database connection closed.")